[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch10-model-compression/03-luong-tu-hoa-bang-tay.ipynb)

# Lượng tử hoá bằng tay: hệ số co giãn, điểm không, và cái giá của từng bit

Lượng tử hoá (quantization) ánh xạ giá trị số thực sang một tập nhỏ các mức nguyên. Chương 10
nhấn mạnh rằng đó không phải "chỉ là làm tròn": đó là một phép ánh xạ có mất mát, và kết quả tuỳ
khoảng cắt, độ chi tiết của hệ số co giãn, và số bit. Sổ tay này làm mọi bước bằng phép toán
tensor thường, không dùng thư viện lượng tử hoá nào, để bạn thấy từng con số đi đâu.

**Bạn sẽ làm:**
1. tính lại napkin math 1.5 của chương: hệ số co giãn và điểm không cho khoảng từ âm 1 tới 3;
2. so dạng đối xứng với dạng bất đối xứng trên một phân phối lệch;
3. đo sai số theo số bit, từ 8 xuống 2;
4. xem một điểm ngoại lai làm hỏng hệ số co giãn chung thế nào, và hai cách chữa;
5. lượng tử hóa sau huấn luyện (post-training quantization, PTQ) một mạng MNIST, theo tầng và theo kênh;
6. hiệu chuẩn và lượng tử hoá giá trị kích hoạt (activation);
7. đếm đúng từng byte của tệp mô hình sau lượng tử hoá;
8. huấn luyện có tính tới lượng tử hóa (quantization-aware training, QAT) với bộ ước lượng thẳng
   (straight-through estimator).

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Napkin math 1.5: hệ số co giãn và điểm không

Phép ánh xạ afin của chương là $x \approx s\,(x_q - z)$, với $s$ là hệ số co giãn (độ dài một
bước) và $z$ là điểm không (số nguyên ứng với số 0 thực). Cho khoảng thực $[\alpha, \beta]$ và
$b$ bit không dấu:

$$s = \frac{\beta - \alpha}{2^b - 1}, \qquad z = \mathrm{round}\!\left(\frac{-\alpha}{s}\right), \qquad
x_q = \mathrm{clamp}\!\left(\mathrm{round}\!\left(\frac{x}{s} + z\right), 0, 2^b - 1\right)$$

Kịch bản của chương: giá trị kích hoạt trải từ âm 1 tới 3, độ chính xác đích là UINT8.

In [ ]:
alpha = -sach(1, nguon="âm 1 · 3 · 8 bit")
beta = sach(3, nguon="âm 1 · 3 · 8 bit")
b = sach(8, nguon="3 · 8 bit")

so_buoc = 2 ** b - 1
theo_sach("độ dài khoảng thực", beta - alpha, sach=4, nguon="8 bit · 4 · 255")
theo_sach("số bước 2^8 − 1", so_buoc, sach=255, nguon="4 · 255")
s = (beta - alpha) / so_buoc
theo_sach("hệ số co giãn s", s, sach=0.0157, nguon="255 · 0.0157")
z_chua_tron = -alpha / s
theo_sach("điểm không trước khi làm tròn", z_chua_tron, sach=63.75, nguon="0.0157 · 63.75")
z = round(z_chua_tron)
theo_sach("điểm không z", z, sach=64, nguon="63.75 · 64")


def luong_tu_afin(x, s, z, b):
    return torch.clamp(torch.round(x / s + z), 0, 2 ** b - 1)


def giai_luong_tu(q, s, z):
    return s * (q - z)


x = torch.tensor([-1.0, -0.5, 0.0, 0.01, 1.0, 2.9, 3.0, 3.5])
q = luong_tu_afin(x, s, z, b)
for xi, qi, xh in zip(x.tolist(), q.tolist(), giai_luong_tu(q, s, z).tolist()):
    print(f"x = {xi:>5.3g}: số nguyên {int(qi):>3}, giải lượng tử hoá {xh:+.4f}, sai số {xh - xi:+.4f}")
assert luong_tu_afin(torch.tensor(0.0), s, z, b) == z and giai_luong_tu(torch.tensor(float(z)), s, z) == 0

# Trong khoảng, sai số làm tròn không bao giờ vượt nửa bước
dat_hat_giong()
x_deu = alpha + (beta - alpha) * torch.rand(100_000)
loi = (giai_luong_tu(luong_tu_afin(x_deu, s, z, b), s, z) - x_deu).abs().max().item()
print(f"sai số lớn nhất trên 100,000 giá trị trong khoảng: {loi:.5f} (nửa bước: {s / 2:.5f})")
assert loi <= s / 2 + 1e-6

Số 0 thực rơi đúng vào số nguyên 64 và quay về đúng 0.0, không lệch chút nào. Chương nói đó là lý
do phần đệm bằng 0 trong CNN được biểu diễn chính xác. Giá trị 3.5 nằm ngoài khoảng nên bị kẹp: sai
số của nó không còn bị chặn bởi nửa bước nữa, và đó là cái giá của khoảng cắt.

## 2. Đối xứng hay bất đối xứng

Dạng đối xứng của chương là $q = \mathrm{round}(x/s)$, với $s$ lấy từ giá trị tuyệt đối lớn nhất,
và số nguyên có dấu trong miền INT8 từ âm 128 tới 127. Nó buộc số 0 thực rơi vào số nguyên 0.
Dạng bất đối xứng dùng điểm không ở phần 1.

### Dự đoán

Đầu ra của ReLU không bao giờ âm. Lượng tử hoá nó bằng dạng đối xứng 4 bit thì bao nhiêu trong 16
mức nguyên được dùng tới?

In [ ]:
Q_MIN, Q_MAX = -sach(128, nguon="âm 128 tới 127"), sach(127, nguon="âm 128 tới 127")


def doi_xung(x, b, s=None):
    """Lượng tử hoá đối xứng b bit rồi giải lượng tử hoá. Trả (giá trị giả, các mức nguyên)."""
    qmax = 2 ** (b - 1) - 1
    s = x.abs().max() / qmax if s is None else s
    qi = torch.clamp(torch.round(x / s), -qmax - 1, qmax)
    return qi * s, qi


def bat_doi_xung(x, b, lo=None, hi=None):
    lo = min(x.min().item(), 0.0) if lo is None else lo     # khoảng phải chứa số 0
    hi = max(x.max().item(), 0.0) if hi is None else hi
    s_ = (hi - lo) / (2 ** b - 1)
    z_ = round(-lo / s_)
    qi = luong_tu_afin(x, s_, z_, b)
    return giai_luong_tu(qi, s_, z_), qi


assert 2 ** (8 - 1) - 1 == Q_MAX and -(2 ** (8 - 1)) == Q_MIN   # INT8 có dấu: đúng miền của sách

dat_hat_giong()
relu = torch.relu(torch.randn(100_000))
cac_phan_phoi = {"ReLU (không âm)": relu, "đều trên [âm 1, 3]": x_deu}
for ten, du_lieu in cac_phan_phoi.items():
    for bit in (8, 4):
        gs, qs = doi_xung(du_lieu, bit)
        ga, qa = bat_doi_xung(du_lieu, bit)
        print(f"{ten:>18}, {bit} bit: đối xứng dùng {qs.unique().numel():>3}/{2 ** bit} mức, "
              f"MSE {(gs - du_lieu).pow(2).mean():.2e} · bất đối xứng dùng {qa.unique().numel():>3}/{2 ** bit} mức, "
              f"MSE {(ga - du_lieu).pow(2).mean():.2e}")

# Hình: các mức 4 bit của hai cách, đặt cạnh biểu đồ tần suất của đầu ra ReLU
print(f"{(relu == 0).float().mean():.1%} đầu ra ReLU bằng đúng 0; hình dưới chỉ vẽ phần dương")
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(relu[relu > 0].numpy(), bins=80, color="0.8")
s4 = relu.max().item() / 7
for qi in range(-8, 8):
    ax.axvline(qi * s4, ymin=0.0, ymax=0.42, color="C0", lw=1.4, ls=":" if qi < 0 else "-")
sa = relu.max().item() / 15
for qi in range(16):
    ax.axvline(qi * sa, ymin=0.58, ymax=1.0, color="C1", lw=1.4)
ax.text(0.03, 0.85, "bất đối xứng 4 bit (trên):\ncả 16 mức nằm trên phần dương",
        color="C1", fontsize=8, va="center", transform=ax.transAxes)
ax.text(0.03, 0.55, "đối xứng 4 bit (dưới): 8 mức âm\nkhông bao giờ được dùng (nét chấm)",
        color="C0", fontsize=8, va="center", transform=ax.transAxes)
ax.set_xlabel("giá trị")
ax.set_ylabel("số giá trị dương")
ax.set_title("Đầu ra ReLU và các mức nguyên của hai cách lượng tử hoá", fontsize=10)
plt.show()

Với đầu ra ReLU, dạng đối xứng phí gần nửa số mức cho những giá trị âm không bao giờ xuất hiện, nên
mỗi bước rộng gấp đôi cần thiết. Đó chính là câu của chương: dạng đối xứng phí miền động khi phân
phối bị lệch. Dạng bất đối xứng dùng đủ mọi mức, đổi lại phải lưu và áp thêm điểm không.

## 3. Mỗi bit đáng bao nhiêu

Với một bộ lượng tử hoá đều, chương nói sai số làm tròn trong khoảng luôn bị chặn trong nửa bước về
mỗi phía. Nếu sai số rải đều trong khoảng ấy, sai số bình phương trung bình là $\Delta^2/12$, với
$\Delta$ là độ dài một bước. Bớt một bit thì $\Delta$ gấp đôi.

### Dự đoán

Bớt một bit thì sai số bình phương trung bình tăng bao nhiêu lần?

In [ ]:
dat_hat_giong()
w = torch.randn(100_000) * 0.05            # trọng số giả lập: phân phối chuẩn quanh 0
cac_bit = [8, 6, 4, 3, 2]
mse, ly_thuyet = [], []
for bit in cac_bit:
    wq, _ = doi_xung(w, bit)
    delta = (w.abs().max() / (2 ** (bit - 1) - 1)).item()
    assert (wq - w).abs().max().item() <= delta / 2 + 1e-7       # bị chặn trong nửa bước
    mse.append((wq - w).pow(2).mean().item())
    ly_thuyet.append(delta ** 2 / 12)
    print(f"{bit} bit: bước {delta:.2e}, MSE đo được {mse[-1]:.2e}, Δ²/12 = {ly_thuyet[-1]:.2e}")

fig, ax = plt.subplots()
ax.semilogy(cac_bit, mse, "o-", label="đo được")
ax.semilogy(cac_bit, ly_thuyet, "--", color="0.4", label="Δ²/12 (sai số rải đều)")
ax.set_xticks(cac_bit)
ax.invert_xaxis()
ax.set_xlabel("số bit")
ax.set_ylabel("sai số bình phương trung bình (thang log)")
ax.set_title("Lượng tử hoá đối xứng theo tầng, trọng số phân phối chuẩn")
ax.legend(loc="upper left")
plt.show()

Ở 8, 6 và 4 bit, đường đo được nằm sát $\Delta^2/12$: mỗi bit bớt đi nhân sai số bình phương lên
khoảng 4 lần, và ở 3 bit vẫn còn như vậy. Ở 2 bit chỉ còn ba mức, phần lớn giá trị rơi vào mức 0, và
sai số không còn rải đều nữa. Với 2 bit đối xứng, các mức chỉ còn âm 1, 0 và 1 nhân với $s$: đó là tam phân, cần ít
nhất 2 bit mỗi giá trị như chương nói.

In [ ]:
_, q2 = doi_xung(w, sach(2, nguon="1 bit · 2 bit"))
print("các mức nguyên ở 2 bit:", q2.unique().tolist())
assert set(q2.unique().tolist()) <= {-1.0, 0.0, 1.0}

## 4. Một điểm ngoại lai

Chương nói phương pháp max dễ bị điểm ngoại lai kéo giãn: khoảng quá rộng thì phí các mức nguyên.
Ta thêm **một** giá trị lớn gấp nhiều lần phần còn lại vào 100,000 trọng số, rồi lượng tử hoá INT8.

### Dự đoán

Với hệ số co giãn chung cho cả tensor, có bao nhiêu phần trăm trọng số bị làm tròn thành đúng 0?

In [ ]:
w_ngoai_lai = w.clone()
w_ngoai_lai[12_345] = 3.0                  # một giá trị, khoảng 60 lần độ lệch chuẩn
k_ngoai_lai = torch.arange(len(w)) == 12_345

ket_qua = {}
# (a) theo tầng, khoảng cắt bằng giá trị tuyệt đối lớn nhất
ket_qua["theo tầng, khoảng = max"], q_a = doi_xung(w_ngoai_lai, 8)
# (b) theo tầng, khoảng cắt ở phân vị 99.99 của |w|: điểm ngoại lai bị kẹp
nguong = torch.quantile(w_ngoai_lai.abs(), 0.9999).item()
ket_qua["theo tầng, khoảng = phân vị 99.99"], _ = doi_xung(w_ngoai_lai, 8, s=nguong / 127)
# (c) theo kênh: 100 kênh, mỗi kênh 1,000 trọng số, mỗi kênh một hệ số co giãn
theo_kenh = w_ngoai_lai.view(100, 1000)
s_kenh = theo_kenh.abs().amax(dim=1, keepdim=True) / 127
ket_qua["theo kênh, khoảng = max"] = (torch.clamp(torch.round(theo_kenh / s_kenh), -128, 127) * s_kenh).flatten()

w_khong, _ = doi_xung(w, 8)
print(f"không có điểm ngoại lai: MSE {(w_khong - w).pow(2).mean():.2e}, "
      f"{(w_khong == 0).float().mean():.2%} trọng số thành 0")
for ten, wq in ket_qua.items():
    e = wq - w_ngoai_lai
    print(f"{ten:>34}: MSE (trừ điểm ngoại lai) {e[~k_ngoai_lai].pow(2).mean():.2e}, "
          f"{(wq[~k_ngoai_lai] == 0).float().mean():.2%} trọng số thành 0, "
          f"sai số ở chính điểm ngoại lai {e[k_ngoai_lai].abs().item():.3f}")

fig, ax = plt.subplots()
khoang = np.logspace(-8, -1, 57)
for (ten, wq), mau in zip(ket_qua.items(), ["C3", "C1", "C2"]):
    e = (wq - w_ngoai_lai)[~k_ngoai_lai].abs().clamp_min(1.01e-8)   # sai số 0 tuyệt đối vào cột đầu
    ax.hist(e.numpy(), bins=khoang, histtype="step", lw=1.6, color=mau, label=ten)
ax.set_xscale("log")
ax.set_xlabel("độ lớn sai số lượng tử hoá của từng trọng số (thang log)")
ax.set_ylabel("số trọng số")
ax.set_title("INT8 với một điểm ngoại lai: biểu đồ tần suất sai số", fontsize=10)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Một giá trị duy nhất đã kéo giãn bước lượng tử hoá cho cả 100,000 giá trị, và phần lớn chúng rơi về
0. Hai cách chữa đều có giá. Cắt ở phân vị cứu phần đông nhưng làm sai lớn chính điểm ngoại lai,
mà một giá trị lớn bất thường có khi lại quan trọng. Hệ số co giãn theo kênh nhốt điểm ngoại lai
trong kênh của nó, đổi lại phải lưu 100 hệ số thay vì 1. Biểu đồ tần suất sai số là kiểu chẩn đoán
bằng hình ảnh chương nhắc tới: nó cho biết sai số trải rộng hay dồn vào một chỗ.

## 5. Lượng tử hóa sau huấn luyện một mạng thật

Mạng 784-256-128-10 học 5 epoch trên 10,000 ảnh MNIST, rồi trọng số được lượng tử hoá đối xứng ở
nhiều số bit, theo hai độ chi tiết: một hệ số co giãn cho cả tầng, hay một hệ số cho mỗi kênh đầu
ra (mỗi hàng của ma trận trọng số). Chỉ trọng số bị lượng tử hoá, giá trị kích hoạt vẫn là FP32.

In [ ]:
import copy

import torch.nn.functional as F
from torchvision import datasets


def nap_mnist(train, n):
    ds = datasets.MNIST(DATA, train=train, download=True)
    x = ds.data[:n].float().div(255).view(-1, 784)
    return (x - 0.1307) / 0.3081, ds.targets[:n]   # chuẩn hoá theo trung bình và độ lệch của MNIST


X, y = nap_mnist(True, 10_000)
X_kt, y_kt = nap_mnist(False, 10_000)


def tao_mlp():
    return torch.nn.Sequential(torch.nn.Linear(784, 256), torch.nn.ReLU(),
                               torch.nn.Linear(256, 128), torch.nn.ReLU(),
                               torch.nn.Linear(128, 10))


def cac_tang(m):
    return [l for l in m if isinstance(l, torch.nn.Linear)]


def huan_luyen(m, so_epoch, lr=1e-3, lan_truyen=None):
    opt = torch.optim.Adam(m.parameters(), lr)
    lan_truyen = lan_truyen or m
    for _ in range(so_epoch):
        thu_tu = torch.randperm(len(X))
        for i in range(0, len(X), 64):
            chon = thu_tu[i:i + 64]
            loss = F.cross_entropy(lan_truyen(X[chon]), y[chon])
            opt.zero_grad()
            loss.backward()
            opt.step()
    return m


def do_chinh_xac(f):
    with torch.no_grad():
        return (f(X_kt).argmax(1) == y_kt).float().mean().item()


dat_hat_giong()
goc = huan_luyen(tao_mlp(), 5)
acc_fp32 = do_chinh_xac(goc)
do_tren_may("độ chính xác FP32", 100 * acc_fp32, "%")

for i, l in enumerate(cac_tang(goc)):
    hang = l.weight.detach().abs().amax(dim=1)
    print(f"tầng {i + 1}: max |w| của các hàng từ {hang.min():.3f} tới {hang.max():.3f}"
          f" (chênh {hang.max() / hang.min():.1f} lần)")

Các hàng của cùng một tầng có khoảng giá trị khác nhau vài lần, đúng điều kiện mà chương nói lượng
tử hoá theo kênh chính xác hơn theo tầng.

### Dự đoán

Chương mô tả một khuôn mẫu định tính thường gặp: một bình nguyên nơi bớt bit gần như không ảnh
hưởng, rồi một vách đá. Với mạng này, vách đá theo tầng nằm ở bao nhiêu bit? Theo kênh thì sao?

In [ ]:
def luong_tu_trong_so(m, bit, theo_kenh):
    m = copy.deepcopy(m)
    qmax = 2 ** (bit - 1) - 1
    with torch.no_grad():
        for l in cac_tang(m):
            w_ = l.weight
            s_ = (w_.abs().amax(dim=1, keepdim=True) if theo_kenh else w_.abs().max()) / qmax
            l.weight.copy_(torch.clamp(torch.round(w_ / s_), -qmax - 1, qmax) * s_)
    return m


cac_bit = [8, 6, 4, 3, 2]
acc_tang = [do_chinh_xac(luong_tu_trong_so(goc, b_, False)) for b_ in cac_bit]
acc_kenh = [do_chinh_xac(luong_tu_trong_so(goc, b_, True)) for b_ in cac_bit]
for b_, a1, a2 in zip(cac_bit, acc_tang, acc_kenh):
    print(f"{b_} bit: theo tầng {a1:.2%} · theo kênh {a2:.2%}")

fig, ax = plt.subplots()
ax.plot(cac_bit, [100 * a for a in acc_tang], "o-", label="theo tầng")
ax.plot(cac_bit, [100 * a for a in acc_kenh], "s-", label="theo kênh")
ax.axhline(100 * acc_fp32, color="0.5", ls="--", lw=1)
ax.annotate("FP32", (8, 100 * acc_fp32), xytext=(4, -14), textcoords="offset points", color="0.4")
ax.set_xticks(cac_bit)
ax.invert_xaxis()
ax.set_ylim(0, 100)
ax.set_xlabel("số bit của trọng số")
ax.set_ylabel("độ chính xác trên tập kiểm tra (%)")
ax.set_title("Lượng tử hóa sau huấn luyện, chỉ trọng số")
ax.legend(loc="lower left")
plt.show()

Hình này có dạng bình nguyên rồi vách đá. Chương nhấn mạnh rằng số bit ở hai ranh giới không phổ
quát: điểm gãy tuỳ mô hình, phân phối từng tầng, bộ lượng tử hoá và cách hiệu chuẩn. Ở đây chỉ cần
đổi từ theo tầng sang theo kênh là vách đá ở 2 bit đã thấp đi hẳn.

## 6. Giá trị kích hoạt và hiệu chuẩn

Trọng số biết trước, còn giá trị kích hoạt thì đổi theo từng đầu vào. Lượng tử hoá tĩnh tính khoảng
cắt trước, bằng một tập hiệu chuẩn nhỏ, rồi cố định nó khi suy luận. Ta chạy 512 ảnh huấn luyện qua
mạng, ghi lại khoảng giá trị đi vào mỗi tầng, rồi chèn một nút lượng tử hoá trước mỗi tầng. Trọng số
lấy INT8 theo kênh ở mọi cấu hình.

In [ ]:
tap_hieu_chuan = X[:512]
vao_tang = []
with torch.no_grad():
    h = tap_hieu_chuan
    for l in goc:
        if isinstance(l, torch.nn.Linear):
            vao_tang.append(h.clone())
        h = l(h)
for i, a in enumerate(vao_tang):
    print(f"vào tầng {i + 1}: từ {a.min():.3f} tới {a.max():.3f}, phân vị 99.9 của giá trị: "
          f"{torch.quantile(a.flatten()[:200_000], 0.999):.3f}")


class NutLuongTu(torch.nn.Module):
    """Lượng tử hoá giá trị kích hoạt với một khoảng cắt cố định, rồi giải lượng tử hoá."""

    def __init__(self, lo, hi, bit, doi_xung_):
        super().__init__()
        self.lo, self.hi, self.bit, self.dx = lo, hi, bit, doi_xung_

    def forward(self, x):
        if self.dx:
            return doi_xung(x, self.bit, s=max(abs(self.lo), abs(self.hi)) / (2 ** (self.bit - 1) - 1))[0]
        return bat_doi_xung(x, self.bit, self.lo, self.hi)[0]


def mang_luong_tu(bit, doi_xung_, phan_vi=None):
    m = luong_tu_trong_so(goc, 8, True)
    tang, k = [], 0
    for l in m:
        if isinstance(l, torch.nn.Linear):
            a = vao_tang[k]
            k += 1
            hi = a.max().item() if phan_vi is None else torch.quantile(a.flatten()[:200_000], phan_vi).item()
            tang.append(NutLuongTu(min(a.min().item(), 0.0), hi, bit, doi_xung_))
        tang.append(l)
    return torch.nn.Sequential(*tang)


for bit in (8, 4):
    for dx in (True, False):
        do_tren_may(f"kích hoạt {bit} bit, {'đối xứng' if dx else 'bất đối xứng'}, khoảng = max",
                    100 * do_chinh_xac(mang_luong_tu(bit, dx)), "%")
do_tren_may("kích hoạt 4 bit, bất đối xứng, khoảng = phân vị 99.9",
            100 * do_chinh_xac(mang_luong_tu(4, False, 0.999)), "%")

Đầu vào của tầng một là ảnh đã chuẩn hoá, lệch hẳn về một phía của 0. Đầu vào hai tầng sau là đầu
ra ReLU, không bao giờ âm. Đó là trường hợp phần 2: ở 4 bit, dạng đối xứng phí một nửa số mức. Chương
nói hiệu chuẩn giảm sai số trên phân phối đã lấy mẫu, nhưng không bảo đảm kết quả trên dữ liệu
triển khai. Ở đây tập hiệu chuẩn và tập kiểm tra đến từ cùng một phân phối, nên điều kiện ấy dễ hơn
thực tế.

## 7. Tệp mô hình nặng bao nhiêu, tính đúng từng byte

Chương viết FP32 dùng 32 bit, FP16 dùng 16, INT8 dùng 8, nên FP32 sang INT8 giảm tải thô 4 lần, và
INT4 giảm 8 lần. Chữ "thô" là quan trọng: hệ số co giãn, độ lệch (bias) và cách đóng gói nằm ngoài
con số ấy. Ở đây độ lệch giữ FP32 và mỗi hệ số co giãn là một số FP32.

### Dự đoán

Tệp INT8 theo kênh của mạng này nhỏ hơn tệp FP32 đúng 4 lần, hơi ít hơn, hay hơi nhiều hơn?

In [ ]:
BIT_FP32, BIT_FP16, BIT_INT8 = (sach(32, nguon="32 · 16 · 8 bit"), sach(16, nguon="32 · 16 · 8 bit"),
                                sach(8, nguon="32 · 16 · 8 bit"))
BIT_INT4 = 4

trong_so = sum(l.weight.numel() for l in cac_tang(goc))
do_lech = sum(l.bias.numel() for l in cac_tang(goc))
so_kenh = sum(l.weight.shape[0] for l in cac_tang(goc))
so_tang = len(cac_tang(goc))


def byte_tep(bit_trong_so, so_he_so):
    return math.ceil(trong_so * bit_trong_so / 8) + 4 * do_lech + 4 * so_he_so


tep = {"FP32": byte_tep(BIT_FP32, 0),
       "INT8 theo tầng": byte_tep(BIT_INT8, so_tang),
       "INT8 theo kênh": byte_tep(BIT_INT8, so_kenh),
       "INT4 theo kênh": byte_tep(BIT_INT4, so_kenh)}
for ten, n in tep.items():
    print(f"{ten:>15}: {n:>9,} byte, nhỏ hơn FP32 {tep['FP32'] / n:.3f} lần")

theo_sach("tải trọng số thô FP32 / INT8", BIT_FP32 / BIT_INT8, sach=4, nguon="4 lần")
theo_sach("tải trọng số thô FP32 / INT4", BIT_FP32 / BIT_INT4, sach=8, nguon="4× · 8×")

# Đóng gói INT4 thật: hai giá trị 4 bit vào một byte, rồi mở ra lại, không mất gì
w0 = goc[0].weight.detach()
s0 = w0.abs().amax(dim=1, keepdim=True) / 7
q4 = (torch.clamp(torch.round(w0 / s0), -8, 7).to(torch.int8) + 8).to(torch.uint8).flatten()  # 0..15
dong_goi = (q4[0::2] << 4) | q4[1::2]
mo_ra = torch.stack([dong_goi >> 4, dong_goi & 0xF], dim=1).flatten()
assert torch.equal(mo_ra, q4)
print(f"tầng 1 ở INT4: {q4.numel():,} giá trị trong {dong_goi.numel():,} byte")

Tỉ số tải thô là 4 và 8, đúng như sách. Tệp thật nhỏ hơn hơi ít hơn thế, vì độ lệch và hệ số co giãn
không co lại. Với mạng nhỏ này phần ấy chỉ là một phần nhỏ, nhưng nó lớn dần khi độ chi tiết mịn hơn:
đó là lý do chương nói mỗi hệ số co giãn thêm vào đều phải được lưu, nạp và áp dụng.

Chương còn nói PTQ và QAT thường nhắm 8 bit hoặc 4 bit. Phần cuối thử xuống thấp hơn nữa.

## 8. Nút lượng tử hoá giả và bộ ước lượng thẳng

Lượng tử hoá giả làm tròn ở lượt truyền xuôi nhưng vẫn trả về số thực. Hàm ấy là một bậc thang:
đạo hàm thật bằng 0 gần như khắp nơi, nên gradient sẽ bị chặn. Bộ ước lượng thẳng coi phép làm tròn
như có đạo hàm bằng 1 trong khoảng cắt, và 0 ngoài nó. Trong PyTorch, cách viết ngắn nhất là
`x + (q(x) − x).detach()`: lượt xuôi thấy `q(x)`, lượt ngược chỉ thấy `x`.

In [ ]:
def luong_tu_gia(x, bit, s_):
    """Lượng tử hoá giả đối xứng với bộ ước lượng thẳng: gradient 1 trong khoảng cắt, 0 ngoài nó."""
    qmax = 2 ** (bit - 1) - 1
    x_kep = torch.clamp(x, -qmax * s_, qmax * s_)            # đạo hàm của clamp: 1 trong, 0 ngoài
    q_ = torch.round(x_kep / s_) * s_
    return x_kep + (q_ - x_kep).detach()


xs = torch.linspace(-1.5, 1.5, 601, requires_grad=True)
ra = luong_tu_gia(xs, 3, 1 / 3)                              # 3 bit, khoảng cắt [−1, 1]
ra.sum().backward()
trong = xs.detach().abs() < 1 - 1e-6
ngoai = xs.detach().abs() > 1 + 1e-6
assert torch.all(xs.grad[trong] == 1) and torch.all(xs.grad[ngoai] == 0)

fig, (a0, a1) = plt.subplots(2, 1, figsize=(7, 4.6), sharex=True)
a0.plot(xs.detach(), ra.detach(), color="C0")
a0.plot(xs.detach(), xs.detach(), color="0.6", ls=":", lw=1)
a0.set_ylabel("lượt xuôi: q(x)")
a0.set_title("Lượng tử hoá giả 3 bit, khoảng cắt từ âm 1 tới 1", fontsize=10)
a1.plot(xs.detach(), xs.grad, color="C1")
a1.set_ylabel("lượt ngược: đạo hàm")
a1.set_yticks([0, 1])
a1.set_xlabel("x")
plt.tight_layout()
plt.show()

### QAT ở 2 bit

Ở 2 bit theo tầng, phần 5 cho thấy PTQ gãy. Bây giờ ta tinh chỉnh mạng 2 epoch với nút lượng tử hoá
giả trên mọi trọng số, rồi lượng tử hoá thật. Một phép so công bằng cần đối chứng: cùng mạng FP32
được học thêm 2 epoch như vậy rồi mới PTQ, để phần tăng không chỉ là nhờ được huấn luyện lâu hơn.

### Dự đoán

QAT kéo được độ chính xác 2 bit về gần FP32 không?

In [ ]:
BIT_QAT = 2


def lan_truyen_qat(m):
    qmax = 2 ** (BIT_QAT - 1) - 1

    def f(x):
        h = x
        for l in m:
            if isinstance(l, torch.nn.Linear):
                s_ = l.weight.detach().abs().max() / qmax         # theo tầng, khoảng = max
                h = F.linear(h, luong_tu_gia(l.weight, BIT_QAT, s_), l.bias)
            else:
                h = l(h)
        return h
    return f


dat_hat_giong(3)
qat = copy.deepcopy(goc)
t0 = time.perf_counter()
huan_luyen(qat, 2, lr=1e-4, lan_truyen=lan_truyen_qat(qat))
do_tren_may("thời gian QAT 2 epoch", time.perf_counter() - t0, "s")

dat_hat_giong(3)
doi_chung = huan_luyen(copy.deepcopy(goc), 2, lr=1e-4)

do_tren_may("PTQ 2 bit theo tầng, mạng gốc", 100 * do_chinh_xac(luong_tu_trong_so(goc, BIT_QAT, False)), "%")
do_tren_may("PTQ 2 bit, đối chứng học thêm 2 epoch FP32", 100 * do_chinh_xac(luong_tu_trong_so(doi_chung, BIT_QAT, False)), "%")
do_tren_may("QAT 2 epoch, rồi lượng tử hoá 2 bit", 100 * do_chinh_xac(luong_tu_trong_so(qat, BIT_QAT, False)), "%")
do_tren_may("FP32 để so", 100 * acc_fp32, "%")

Nếu dòng QAT cao hơn hẳn dòng đối chứng, phần chênh ấy đến từ việc mạng đã học **cùng với** nhiễu
lượng tử hoá, chứ không từ hai epoch thêm. Chương đặt điều này đúng chỗ của nó: bắt đầu bằng PTQ và
đo, chỉ khi PTQ hụt ngưỡng sản phẩm thì mới trả thêm chi phí kỹ thuật của QAT, và lợi thế của QAT tuỳ
mô hình, bộ lượng tử hoá và quy trình huấn luyện.

## Thử thêm

1. Ở phần 1, đổi `b` thành 4. Hệ số co giãn và điểm không bây giờ là bao nhiêu, và số 0 thực còn
   được biểu diễn chính xác không? (Các dấu ✗ khi đó là điều được chờ đợi: sách tính với 8 bit.)
2. Ở phần 4, đổi `3.0` thành `0.3`. Từ độ lớn nào thì điểm ngoại lai thôi làm hại hệ số co giãn chung?
3. Ở phần 8, đặt `BIT_QAT = 3`, rồi thử lượng tử hoá theo kênh trong `lan_truyen_qat`. QAT còn đáng
   công không khi PTQ đã gần đủ?

## Tóm lại

* Hệ số co giãn là độ dài một bước, điểm không là số nguyên ứng với số 0 thực, và dạng bất đối xứng
  dùng đủ mọi mức khi phân phối lệch, như đầu ra ReLU.
* Mỗi bit bớt đi nhân sai số bình phương lên khoảng 4 lần, cho tới khi còn quá ít mức để sai số rải đều.
* Một điểm ngoại lai đủ để kéo giãn hệ số co giãn chung cho cả tensor. Khoảng cắt ở phân vị và hệ số
  theo kênh là hai cách chữa, mỗi cách một giá.
* Tải thô giảm đúng 4 lần ở INT8 và 8 lần ở INT4, còn tệp thật nhỏ hơn hơi ít hơn vì độ lệch và hệ số
  co giãn không co lại.
* QAT với bộ ước lượng thẳng có thể kéo về phần mà PTQ đánh mất ở số bit thấp, nhưng phải so với một
  đối chứng được huấn luyện cùng thời gian.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Nén mô hình — đổi phần dư thừa lấy ràng buộc triển khai](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch10-model-compression/), dựng từ chương
[*Model Compression*](https://mlsysbook.ai/vol1/model_compression/model_compression.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.